# 06 - Evaluation, confusion-matrix analysis and paper comparison

**What this notebook does**
1. Loads the three results files and shows them side by side.
2. Reloads the saved MiniConvNet checkpoints and produces confusion matrices plus the
   tumour-vs-healthy / subtype-vs-subtype breakdown (LESSON 9) - this is the explanation for any gap
   versus the paper, not a decorative plot.
3. Builds the model-comparison figure and the paper-vs-replication table.
4. Writes `outputs/reports/comparison.md` containing the computed numbers and clearly-marked
   placeholders for the narrative you write yourself.

**What must already exist**: notebooks 00 and 02-05. If a checkpoint is missing this notebook skips
that model with a warning rather than failing - re-run the corresponding training notebook.

**What "looks right"**: `results_table.csv` has one row per model with no `INVALID_collapsed` rows;
the confusion matrices show `normal` cleanly separated with the residual errors concentrated among
the three tumour subtypes.

In [ ]:
import sys, os
sys.path.append(os.path.abspath(".."))

from src.config import *
from src.data_utils import resolve_data_root

ensure_dirs()
print('data root:', resolve_data_root())
print('models dir:', MODELS_DIR)

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import tensorflow as tf

from src.data_utils import load_split, make_split_datasets
from src.train_utils import set_global_seeds, checkpoint_path, load_history, run_name_for
from src.evaluate_utils import (predict, compute_metrics, per_class_report, confusion,
                               plot_confusion_matrix, tumor_vs_subtype_breakdown,
                               interpret_breakdown, detect_collapse, load_results, valid_only,
                               format_mean_std)

set_global_seeds(SEED)
pd.set_option('display.width', 160)

## 1. The three results files

**Looks right**: `results_table.csv` = one row per model (canonical); `experiments_log.csv` = every
debugging/exploratory run with a `config_note`; `ablation_dropout.csv` = exactly 2 rows.

In [ ]:
canon = load_results('canonical')
exp = load_results('experiment')
abl = load_results('ablation')

print(f'results_table.csv    : {len(canon)} rows')
print(f'experiments_log.csv  : {len(exp)} rows')
print(f'ablation_dropout.csv : {len(abl)} rows')
print()
if len(canon):
    print(canon[['model', 'arch_variant', 'split_variant', 'params', 'accuracy',
                 'accuracy_std', 'f1_macro', 'n_runs', 'status']].round(4).to_string(index=False))
else:
    print('results_table.csv is empty - run notebooks 04 and 05 first.')

In [ ]:
# Duplicate models in the canonical table would break "one row per model".
if len(canon):
    dupes = canon['model'][canon['model'].duplicated()].tolist()
    print('duplicate model rows:', dupes if dupes else 'none - good')
    bad = canon[canon['status'] != VALID_TAG]
    print('collapsed rows in the canonical table:',
          bad['model'].tolist() if len(bad) else 'none - good')

## 2. Model comparison (valid rows only)

Collapsed rows are excluded before ranking - a dead network must never top the table.

In [ ]:
ranked = valid_only(canon).sort_values('accuracy', ascending=False) if len(canon) else canon
if len(ranked):
    print(ranked[['model', 'params', 'accuracy', 'accuracy_std', 'f1_macro',
                  'cohen_kappa', 'mcc']].round(4).to_string(index=False))
else:
    print('nothing to rank yet')

In [ ]:
if len(ranked):
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))

    err = ranked['accuracy_std'].fillna(0).values
    axes[0].barh(ranked['model'], ranked['accuracy'], xerr=err, color='#4c78a8')
    axes[0].set_xlabel('test accuracy')
    axes[0].set_title('Accuracy by model (error bars = CV std where available)')
    axes[0].invert_yaxis()
    axes[0].grid(axis='x', alpha=0.3)

    axes[1].scatter(ranked['params'], ranked['accuracy'], s=60, color='#e45756')
    for _, r in ranked.iterrows():
        axes[1].annotate(r['model'], (r['params'], r['accuracy']), fontsize=7,
                         xytext=(4, 4), textcoords='offset points')
    axes[1].set_xscale('log')
    axes[1].set_xlabel('parameters (log scale)')
    axes[1].set_ylabel('test accuracy')
    axes[1].set_title('Accuracy vs model size')
    axes[1].grid(alpha=0.3)

    fig.tight_layout()
    out = FIGURES_DIR / 'model_comparison.png'
    fig.savefig(out, dpi=150, bbox_inches='tight')
    plt.show()
    print('saved', out)

## 3. Reload the saved MiniConvNet checkpoints

Uses the single-run checkpoints written by notebook 02 (`models/miniconvnet_<arch>_<split>.keras`).

**Looks right**: every checkpoint listed as `found`. A missing one means that run has not been
trained yet in this session - re-run notebook 02.

In [ ]:
targets = [(a, s) for s in SPLIT_VARIANTS for a in ARCH_VARIANTS]
available = []
for arch, split in targets:
    name = run_name_for('miniconvnet', arch, split)
    p = checkpoint_path(name)
    status = 'found' if p.exists() else 'MISSING - re-run notebook 02'
    print(f'{name:34s} {status}')
    if p.exists():
        available.append((arch, split, name, p))
print('\navailable checkpoints:', len(available))

In [ ]:
# Test datasets, one per split variant (built once and reused).
test_sets = {}
for split in SPLIT_VARIANTS:
    sdf = load_split(split)
    _, _, test_ds, frames = make_split_datasets(sdf)
    test_sets[split] = {'ds': test_ds, 'frame': frames['test']}
    print(f'{split}: {len(frames["test"])} test images')

## 4. Confusion matrices and the tumour-vs-subtype breakdown (LESSON 9)

The headline accuracy hides two very different tasks. `binary_tumor_vs_healthy_accuracy` is
tumour detection; `subtype_accuracy_all_tumors` is telling the three tumour subtypes apart. In the
earlier attempt detection was ~97% while subtype discrimination was much worse, with adenocarcinoma
absorbing most of the confusions - that difference, not a training bug, was the gap versus the
paper.

**Looks right**: a large positive gap between the binary and subtype numbers, and a confusion matrix
whose off-diagonal mass sits inside the 3x3 tumour block.

In [ ]:
analyses = {}
for arch, split, name, path in available:
    print('=' * 72)
    print(name)
    model = tf.keras.models.load_model(path)
    y_true, y_pred, y_prob = predict(model, test_sets[split]['ds'])
    metrics = compute_metrics(y_true, y_pred, y_prob)
    collapse = detect_collapse(kappa=metrics['cohen_kappa'], mcc=metrics['mcc'], y_pred=y_pred)
    breakdown = tumor_vs_subtype_breakdown(y_true, y_pred)

    print('metrics :', {k: round(v, 4) for k, v in metrics.items()})
    print('status  :', collapse['status'])
    print('\n' + interpret_breakdown(breakdown))
    print('\nper-class report:')
    print(per_class_report(y_true, y_pred).round(4))

    plot_confusion_matrix(y_true, y_pred, name)
    plot_confusion_matrix(y_true, y_pred, name, normalize=True)

    analyses[name] = {'arch': arch, 'split': split, 'metrics': metrics,
                      'breakdown': breakdown, 'collapse': collapse,
                      'cm': confusion(y_true, y_pred),
                      'interpretation': interpret_breakdown(breakdown)}
    tf.keras.backend.clear_session()
    print()

In [ ]:
if analyses:
    bd = pd.DataFrame([
        {'model': k, 'arch': v['arch'], 'split': v['split'],
         'overall_accuracy': v['metrics']['accuracy'],
         'tumor_vs_healthy': v['breakdown']['binary_tumor_vs_healthy_accuracy'],
         'subtype_accuracy': v['breakdown']['subtype_accuracy_all_tumors'],
         'most_over_predicted': v['breakdown']['most_over_predicted_class'],
         'status': v['collapse']['status']}
        for k, v in analyses.items()])
    bd['detection_minus_subtype'] = (bd['tumor_vs_healthy'] - bd['subtype_accuracy']).round(4)
    print(bd.round(4).to_string(index=False))
else:
    print('No checkpoints analysed - run notebook 02 first.')

## 5. Faithful vs clean: how much of the accuracy was leakage?

The `clean` variant removes the duplicate-driven train/test leakage. The difference between the two
is a measure of how much of the `faithful` number the leakage was worth.

**Reminder**: `clean` results are a robustness / generalisation experiment. Never present them as a
direct replication of the paper.

In [ ]:
if analyses:
    for arch in ARCH_VARIANTS:
        f = analyses.get(run_name_for('miniconvnet', arch, 'faithful'))
        c = analyses.get(run_name_for('miniconvnet', arch, 'clean'))
        if f and c:
            d = f['metrics']['accuracy'] - c['metrics']['accuracy']
            print(f"{arch:8s}: faithful={f['metrics']['accuracy']:.4f} "
                  f"clean={c['metrics']['accuracy']:.4f} drop={d:+.4f}")
        else:
            print(f'{arch:8s}: need both faithful and clean checkpoints for this comparison')

## 6. Paper vs replication

`PAPER_CLAIMS` below holds only what the **paper itself reports** - nothing is invented and nothing
is pre-filled on the replication side. Add the paper's per-baseline numbers as you read them off the
published table; entries left as `None` simply print as blank.

In [ ]:
# Values reported IN THE PAPER (Baqir et al., Sci Rep 16:12985, 2026).
# Fill in the remaining entries from the published tables; leave None if not reported.
PAPER_CLAIMS = {
    'MiniConvNet': {'accuracy': 0.96, 'params': 500_000},
    'ResNet50': {'accuracy': None, 'params': None},
    'VGG16': {'accuracy': None, 'params': None},
    'MobileNetV3Small': {'accuracy': None, 'params': None},
    'EfficientNetV2B0': {'accuracy': None, 'params': None},
}

def our_number(model_name):
    if not len(canon):
        return None
    hit = canon[canon['model'] == model_name]
    if not len(hit):
        return None
    r = hit.iloc[0]
    if r['status'] != VALID_TAG:
        return f"{r['status']}"
    if pd.notna(r.get('accuracy_std')) and r.get('n_runs', 1) and r['n_runs'] > 1:
        return format_mean_std(r['accuracy'], r['accuracy_std'])
    return f"{r['accuracy']:.4f}"

# Which of our rows corresponds to each paper row.
OUR_ROW_FOR = {
    'MiniConvNet': 'MiniConvNet-flatten (5-fold CV)',   # paper-sized variant, CV mean+/-std
    'ResNet50': 'ResNet50',
    'VGG16': 'VGG16',
    'MobileNetV3Small': 'MobileNetV3Small',
    'EfficientNetV2B0': 'EfficientNetV2B0',
}

comparison = pd.DataFrame([
    {'model': k,
     'paper_accuracy': v['accuracy'],
     'our_accuracy': our_number(OUR_ROW_FOR[k]),
     'paper_params': v['params'],
     'our_params': (canon.loc[canon['model'] == OUR_ROW_FOR[k], 'params'].iloc[0]
                    if len(canon) and (canon['model'] == OUR_ROW_FOR[k]).any() else None)}
    for k, v in PAPER_CLAIMS.items()])
print(comparison.to_string(index=False))
print('\nAlso report, for MiniConvNet: the gap variant (5-fold CV) and both clean-split runs '
      '- see experiments_log.csv.')

## 7. Write `outputs/reports/comparison.md`

The report contains the computed numbers plus `TODO:` markers where a human has to write the
interpretation. Nothing narrative is auto-invented.

In [ ]:
lines = [
    '# Paper vs replication',
    '',
    'Paper: Baqir, M.A., Qayyum, S., Ashfaq, N. et al. "A lightweight CNN for enhanced '
    'non-small cell lung cancer classification using CT scan image." '
    'Scientific Reports 16, 12985 (2026). DOI: 10.1038/s41598-026-41401-w',
    '',
    '## Headline comparison',
    '',
    '```',
    comparison.to_string(index=False),
    '```',
    '',
    'MiniConvNet is reported as the 5-fold cross-validation mean +/- std on the faithful '
    '(paper-comparable) split, not a single best run.',
    '',
    '## Canonical results table',
    '',
    '```',
    (canon.to_string(index=False) if len(canon) else '(empty - run notebooks 04 and 05)'),
    '```',
    '',
    '## Dropout ablation',
    '',
    '```',
    (abl.to_string(index=False) if len(abl) else '(empty - run notebook 03)'),
    '```',
    '',
    '## Confusion-matrix explanation of the gap',
    '',
]
for name, a in analyses.items():
    lines += [
        f'### {name} ({a["arch"]} head, {a["split"]} split)', '',
        f'- overall accuracy: {a["metrics"]["accuracy"]:.4f}',
        f'- tumour-vs-healthy accuracy: {a["breakdown"]["binary_tumor_vs_healthy_accuracy"]:.4f}',
        f'- subtype accuracy (true tumour samples): {a["breakdown"]["subtype_accuracy_all_tumors"]:.4f}',
        f'- most over-predicted class: {a["breakdown"]["most_over_predicted_class"]}',
        f'- status: {a["collapse"]["status"]}', '',
        a['interpretation'], '',
        '```', str(a['cm']), '```', '',
    ]
lines += [
    '## Interpretation (write these yourself)',
    '',
    '- TODO: does the replication reproduce the paper\'s 96% on the faithful split? By how much '
    'does it differ?',
    '- TODO: how much of the faithful-split accuracy is attributable to duplicate-driven '
    'train/test leakage (compare with the clean-split runs)?',
    '- TODO: which architecture variant (gap vs flatten) is closer to the paper, and does the '
    'parameter-count ambiguity change the conclusion?',
    '- TODO: is the residual error a tumour-detection problem or a subtype-discrimination problem, '
    'given the breakdown above?',
    '- TODO: what would you change to close the remaining gap?',
]

report_path = REPORTS_DIR / 'comparison.md'
report_path.write_text('\n'.join(lines))
print('wrote', report_path)
print()
print('\n'.join(lines[:20]))

In [ ]:
print('artefacts:')
print('  figures :', FIGURES_DIR)
print('  history :', HISTORY_DIR)
print('  reports :', REPORTS_DIR)
print('  tables  :', RESULTS_TABLE_CSV.name, ',', EXPERIMENTS_LOG_CSV.name, ',', ABLATION_CSV.name)
print('\noptional extension: 07_gradcam_optional.ipynb')